# GPT Homework: Parallel Attention and Addition

This notebook tracks two Karpathy GPT homework exercises:

- **EX1:** merge the separate `Head` and `MultiHeadAttention` classes so all attention heads are computed together with tensor operations.
- **EX2:** train a small decoder-only Transformer on synthetic addition problems, using examples like `a+b=c` as the language modeling task.

The goal is to understand the Transformer mechanics directly: token/position embeddings, causal self-attention, multi-head attention, feed-forward blocks, residual connections, layer norm, loss masking, and autoregressive generation.


In [1]:
import math
import inspect
from dataclasses import dataclass
import random
import os
import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:

# Check for XLA (TPU), then CUDA (GPU), then CPU
if 'COLAB_TPU_ADDR' in os.environ:
    # Only import torch_xla when running on TPU
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
elif torch.cuda.is_available():
    device = 'cuda'
elif torch.mps.is_available():
  device = 'mps'
else:
    device = 'cpu'

print(device)

mps


In [3]:
# hyperparameters
batch_size = 16 # B how many independent sequences will we process in parallel?
block_size = 12 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3

eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0

g = torch.Generator().manual_seed(4242)


In [4]:
chars = sorted(['+','0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=',' '])
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string


In [5]:
decimal_place = 1000 # 10 is single digit, 100 is 2, 1000 is 3 etc
n_data = int(math.pow(decimal_place,2)) # 100 x 100 = 10000 total permutations
n_val = int(0.1 * n_data)


pairs = []
for a in range(decimal_place):
  for b in range(decimal_place):
    pairs.append((a,b))

random.shuffle(pairs)

val_pairs = pairs[:n_val]
train_pairs = pairs[n_val:]

In [54]:
def get_batch(split):
    batch_pairs = train_pairs if split == 'train' else val_pairs
    ix = torch.randint(len(batch_pairs), (batch_size,)).tolist()

    rows = []
    for i in ix:
        a, b = batch_pairs[i]
        c = a + b
        c_str = f"{c:04d}"[::-1]
        s = f"{a:03d}+{b:03d}={c_str} "
        rows.append(torch.tensor(encode(s)))


    full = torch.stack(rows)         # (B, 9)
    x = full[:, :block_size]                  # (B, 8)
    y = full[:, 1:block_size+1].clone()
    y[:, 0:7] = -1

    x, y = x.to(device), y.to(device)
    return x, y

In [55]:
a = torch.randn(4,4)
a

tensor([[ 0.6718, -1.2376,  0.3639,  0.8044],
        [-0.6449, -0.2553,  1.3931, -1.0646],
        [-1.0323, -1.3815, -0.7883,  0.5066],
        [-0.2335,  1.0768,  0.1391, -0.4832]])

In [69]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()#?
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# class Head(nn.Module):
#     """ one head of self-attention """

#     def __init__(self, head_size,n_head):
#         super().__init__()
#         self.key = nn.Linear(n_embd, n_head * head_size, bias=False) 
#         self.query = nn.Linear(n_embd, n_head * head_size, bias=False)  # 64,16
#         self.value = nn.Linear(n_embd, n_head * head_size, bias=False) 
#         self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) #??

#         self.dropout = nn.Dropout(dropout) # ?

#     def forward(self, x):
#         B,T,C = x.shape #16,11,64
        
#         k = self.key(x)   # (B,T,C)
        
#         q = self.query(x) # (B,T,C)

#         v = self.value(x) # (B,T,C)

#         k = k.view(B,T,n_head, head_size)
#         q = q.view(B,T,n_head, head_size)
#         v = v.view(B,T,n_head, head_size)

#         # (B,n_head,T, head_size)
#         k = k.transpose(-3,-2)
#         q = q.transpose(-3,-2)
#         v = v.transpose(-3,-2)

        
        
#         # compute attention scores ("affinities")
#         wei = q @ k.transpose(-2,-1) * head_size**-0.5 # (B,T,n_head, head_size) @ (B,n_head,T, head_size) -> (B, n_head, T, T)

#         # wei now contains the 'affinities' between q and k, for example if the query is something like I am looking for an verb, then the key with 
#         # highest affinity will be the one that contains a verb.
        
#         wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
#         # masking because we are not allowed to have future info aggregated from future tokens (those comming from y)
#         # this is what makes this decoder. 

        
#         wei = F.softmax(wei, dim=-1) # (B, n_head, T, T)
#         # exp(-inf) = 0

        
#         wei = self.dropout(wei)
        
        
#         # perform the weighted aggregation of the values
        
#         out = wei @ v # (B, n_head, T, T) @ (B,n_head,T, head_size) -> (B, n_head, T, head_size)
#         # we aggregate over v not x directly. 

#         out = out.transpose(-3,-2) #(B, T,n_head,  head_size)
#         out = out.reshape(B,T, n_head * head_size)
        
#         return out

class MultiHeadAttention(nn.Module): 
    """ multiple heads of self-attention in parallel """

    def __init__(self,n_head,head_size):
        super().__init__()

        # self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    

        # super().__init__()
        self.key = nn.Linear(n_embd, n_head * head_size, bias=False) 
        self.query = nn.Linear(n_embd, n_head * head_size, bias=False)  # 64,16
        self.value = nn.Linear(n_embd, n_head * head_size, bias=False) 
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) #??
        # self.dropout = nn.Dropout(dropout) # ?
        # print(head_size)
        self.n_head = n_head
        self.head_size = head_size

        print(self.head_size)
        

    def forward(self, x):

        n_head = self.n_head
        head_size = self.head_size

        B,T,C = x.shape #16,11,64
        
        k = self.key(x)   # (B,T,C)
        
        q = self.query(x) # (B,T,C)

        v = self.value(x) # (B,T,C)

        k = k.view(B,T,n_head,head_size)
        q = q.view(B,T,n_head,head_size)
        v = v.view(B,T,n_head,head_size)

        # (B,n_head,T, head_size)
        k = k.transpose(-3,-2)
        q = q.transpose(-3,-2)
        v = v.transpose(-3,-2)

       
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * head_size**-0.5 # (B,T,n_head, head_size) @ (B,n_head,T, head_size) -> (B, n_head, T, T)

        # wei now contains the 'affinities' between q and k, for example if the query is something like I am looking for an verb, then the key with 
        # highest affinity will be the one that contains a verb.
        
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        # masking because we are not allowed to have future info aggregated from future tokens (those comming from y)
        # this is what makes this decoder. 

        wei = F.softmax(wei, dim=-1) # (B, n_head, T, T)
        # exp(-inf) = 0

        wei = self.dropout(wei)
        
        # perform the weighted aggregation of the values
        
        out = wei @ v # (B, n_head, T, T) @ (B,n_head,T, head_size) -> (B, n_head, T, head_size)
        # we aggregate over v not x directly. 

        out = out.transpose(-3,-2) #(B, T,n_head,  head_size)
        out = out.reshape(B,T, n_head * head_size)

        #----
        
        # out = torch.cat([h(x) for h in self.heads], dim=-1)
        #---
        out = self.dropout(self.proj(out))
        return out



class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),)

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # the x being added, is it for residual connections?
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module): #?

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        
        
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape #(B,T)

        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C) #?
        
        
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets, ignore_index=-1)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond) #  loss is ignored here
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
# context = torch.zeros((1, 1), dtype=torch.long, device=device)
# print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

16
16
16
16
0.201741 M parameters
step 0: train loss 2.7781, val loss 2.7792
step 100: train loss 1.4153, val loss 1.4173
step 200: train loss 1.3986, val loss 1.3991
step 300: train loss 1.3907, val loss 1.3903
step 400: train loss 1.3868, val loss 1.3889
step 500: train loss 1.3799, val loss 1.3795
step 600: train loss 1.3529, val loss 1.3585
step 700: train loss 1.1520, val loss 1.1584
step 800: train loss 0.8707, val loss 0.8683
step 900: train loss 0.4654, val loss 0.4602
step 1000: train loss 0.3582, val loss 0.3639
step 1100: train loss 0.2568, val loss 0.2500
step 1200: train loss 0.1391, val loss 0.1439
step 1300: train loss 0.0702, val loss 0.0682
step 1400: train loss 0.1198, val loss 0.1201
step 1500: train loss 0.0607, val loss 0.0579
step 1600: train loss 0.0595, val loss 0.0570
step 1700: train loss 0.0236, val loss 0.0251
step 1800: train loss 0.0090, val loss 0.0103
step 1900: train loss 0.1293, val loss 0.1317
step 2000: train loss 0.1040, val loss 0.0967
step 2100: t

In [71]:
@torch.no_grad()
def generate_addition(model, a, b, max_new_tokens=5):
    model.eval()

    # prompt is only the problem, no answer
    prompt = f"{a:03d}+{b:03d}="
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ = model(idx_cond)

        # greedy decoding: choose most likely next token
        logits = logits[:, -1, :]
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        idx = torch.cat((idx, idx_next), dim=1)

    out = decode(idx[0].tolist())

    # model predicts reversed answer digits, e.g. 12+34=640 means 046 => 46
    pred_reversed = out.split("=")[1]
    pred_normal = pred_reversed[::-1]

    return {
        "prompt": prompt,
        "raw_output": out,
        "pred_reversed": pred_reversed,
        "pred_answer": int(pred_normal),
        "true_answer": a + b,
        "correct": int(pred_normal) == a + b,
    }

In [80]:
a,b = random.randint(0,999),random.randint(0,999)
model.eval()

max_new_tokens=5

# prompt is only the problem, no answer
prompt = f"{a:03d}+{b:03d}="
idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

for _ in range(max_new_tokens):
    idx_cond = idx[:, -block_size:]
    logits, _ = model(idx_cond)

    # greedy decoding: choose most likely next token
    logits = logits[:, -1, :]
    idx_next = torch.argmax(logits, dim=-1, keepdim=True)

    idx = torch.cat((idx, idx_next), dim=1)

out = decode(idx[0].tolist())

# model predicts reversed answer digits, e.g. 12+34=640 means 046 => 46
pred_reversed = out.split("=")[1]
pred_normal = pred_reversed[::-1]

int(pred_normal) == a + b


True

In [73]:
a = torch.rand(2,3,4)
a

tensor([[[0.3273, 0.8105, 0.6311, 0.9807],
         [0.6663, 0.7519, 0.4771, 0.7838],
         [0.9084, 0.0886, 0.5681, 0.3389]],

        [[0.8590, 0.6856, 0.2998, 0.7558],
         [0.8112, 0.3109, 0.8711, 0.6761],
         [0.0292, 0.0820, 0.3708, 0.3649]]])

In [74]:
a.transpose(0,1) #3,2,4

tensor([[[0.3273, 0.8105, 0.6311, 0.9807],
         [0.8590, 0.6856, 0.2998, 0.7558]],

        [[0.6663, 0.7519, 0.4771, 0.7838],
         [0.8112, 0.3109, 0.8711, 0.6761]],

        [[0.9084, 0.0886, 0.5681, 0.3389],
         [0.0292, 0.0820, 0.3708, 0.3649]]])

In [26]:
a.transpose(0,2) #4,3,2

tensor([[[0.1539, 0.3185],
         [0.3841, 0.6216],
         [0.6781, 0.9251]],

        [[0.1544, 0.3850],
         [0.6385, 0.0333],
         [0.4381, 0.8705]],

        [[0.8551, 0.9801],
         [0.4427, 0.9452],
         [0.0454, 0.1354]],

        [[0.1400, 0.3213],
         [0.1319, 0.8653],
         [0.1222, 0.7680]]])

In [27]:
a.transpose(0,-2) # 3,2,4

tensor([[[0.1539, 0.1544, 0.8551, 0.1400],
         [0.3185, 0.3850, 0.9801, 0.3213]],

        [[0.3841, 0.6385, 0.4427, 0.1319],
         [0.6216, 0.0333, 0.9452, 0.8653]],

        [[0.6781, 0.4381, 0.0454, 0.1222],
         [0.9251, 0.8705, 0.1354, 0.7680]]])